In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier

from sklearn.neighbors import KNeighborsClassifier

from sklearn.naive_bayes import GaussianNB

from sklearn.svm import SVC

import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv("processed_logs.csv")
df.head()

,LineId,Date,Time,Pid,Level,Component,EventId,timestamp,hour,content_length,template_length,level_encoded,component_encoded,event_encoded
0,1,81109,203615,148,INFO,dfs.DataNode$PacketResponder,E10,2008-11-09 20:36:15,20,61,49,0,3,1
1,2,81109,203807,222,INFO,dfs.DataNode$PacketResponder,E10,2008-11-09 20:38:07,20,64,49,0,3,1
2,3,81109,204005,35,INFO,dfs.FSNamesystem,E6,2008-11-09 20:40:05,20,121,88,0,5,10
3,4,81109,204015,308,INFO,dfs.DataNode$PacketResponder,E10,2008-11-09 20:40:15,20,63,49,0,3,1
4,5,81109,204106,329,INFO,dfs.DataNode$PacketResponder,E10,2008-11-09 20:41:06,20,64,49,0,3,1


In [3]:
print("Shape:",df.shape)
print()
df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )
print(df.columns)

Shape: (2000, 14)

Index(['lineid', 'date', 'time', 'pid', 'level', 'component', 'eventid',
       'timestamp', 'hour', 'content_length', 'template_length',
       'level_encoded', 'component_encoded', 'event_encoded'],
      dtype='str')


In [4]:
#INFO → Normal → 0
#WARN → Risk → 1

In [5]:
df["target"]=(df["Level"]=="WARN").astype(int)
df["target"].value_counts()

KeyError: 'Level'

In [ ]:
features=["Pid","content_length","template_length","component_encoded"]
X=df[features]
y=df["target"]
X.head()

In [ ]:
X_train,X_test,y_train,y_test=(
    train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
    )
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
models={
"Logistic":
LogisticRegression(
max_iter=3000,
class_weight="balanced"
),
"DecisionTree":
DecisionTreeClassifier(
class_weight="balanced"
),
"RandomForest":
RandomForestClassifier(
n_estimators=200,
class_weight="balanced",
random_state=42
),
"KNN": KNeighborsClassifier(),
"NaiveBayes": GaussianNB(),
"SVM":
SVC(
class_weight="balanced"
)
}

In [ ]:
results=[]
for name,model in models.items():
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    results.append([
        name,
        accuracy_score(y_test,pred),
        precision_score(y_test,pred,zero_division=0),
        recall_score(y_test,pred,zero_division=0),
        f1_score(y_test,pred,zero_division=0)
    ])
results_df=pd.DataFrame(
results,
columns=["Classifier","Accuracy","Precision","Recall","F1"]
)
results_df

In [ ]:
results_df=(results_df.sort_values("F1",ascending=False))
results_df

In [ ]:
plt.figure(figsize=(8,4))
plt.bar(
    results_df["Classifier"],
    results_df["F1"]*100
)
plt.xticks(rotation=30)
plt.ylabel("F1 Score (%)")
plt.xlabel("Classifier")
plt.title("Classifier Comparison")
plt.show()

In [ ]:
best=(results_df.iloc[0])
print()
print("===== BEST MODEL =====")
print()
print(best)

In [ ]:
results_df.to_csv("model_results.csv",index=False)
print("Saved")

Conclusion: 

Logistic / Naive Bayes / SVM → catch nearly every warning (Recall = 1) but raise many false alarms → low precision.
KNN → moderate overall.
Random Forest → very strong balance.
Decision Tree → best overall balance here.